# Surrogate Modelling with bdopt

Surrogate modelling or metamodelling is a way to emulate physics-based building simulation models [[1]].

bdopt integrates building simulation and surrogate modelling using:
+ EnergyPlus model editing and execution via evaluators (`EnergyPlusEvaluator`)
+ sampling toolboxes (`bdopt.sampling`, built on [scipy](https://docs.scipy.org/doc/scipy/reference/stats.qmc.html))
+ machine learning toolboxes like [scikit-learn](https://scikit-learn.org/stable/) (`bdopt.surrogate`)
+ the `bdopt` module to simplify the interconnection of the above items.

[1]: https://www.sciencedirect.com/science/article/pii/S0378778819302877

## Surrogate Modelling workflow

The figure below shows the workflow: from the IDF, the problem (parameters and objectives) and the weather file, the evaluator produces the sample dataset; the dataset is used to train a surrogate model, which is then explored and optimised.

![Image](image/flow_diagram.PNG)

# Surrogate Modelling Work Flow

+ define a problem consisting of parameters to vary with ranges and an objective function
+ sample the problem space
+ define an Evaluator by linking the problem to an EnergyPlus model
+ run the Evaluator for the samples
+ train a surrogate model over the samples
+ explore the design space using an interactive plot that queries the surrogate model

In [ ]:
%matplotlib inline
import time
from pathlib import Path

import numpy as np
import pandas as pd
from ipywidgets import FloatSlider, interact
from matplotlib import pyplot as plt
from sklearn import linear_model, pipeline
from sklearn.preprocessing import StandardScaler

from bdopt import (EnergyPlus, EnergyPlusEvaluator, FieldSelector, FunctionEvaluator, IntegerParameter,
                   Parameter, Problem, RangeParameter, load_idf, sampling, viz)
from window_to_wall import wwr      # the window-to-wall ratio as a design variable: see Besos_overview.ipynb

## Building model
We load an EnergyPlus model.

In [ ]:
idf = load_idf('test.idf')
epw_file = 'ITA_Torino.160590_IWEC.epw'
idf

## Parameters and Objectives
We start by defining `Parameters` that govern how we would like to modify the building.
Here they are the thickness of the insulation layer and the window-to-wall ratio.

(The previous toolbox built the first one with `expand_plist`, which finds the class of an object by its name. bdopt has no such short-cut: you write the class, as below. The window-to-wall ratio is a geometry edit, written as a `GenericSelector` in `window_to_wall.py`, see `Besos_overview.ipynb`.)

In [ ]:
parameters = [
    Parameter(FieldSelector('Material', 'XPS Extruded Polystyrene  - CO2 Blowing_0041', 'Thickness'),
              RangeParameter(0.001, 0.3), name='Insulation Thickness'),      # [m]
    wwr(0.1, 0.8),                                                           # window-to-wall ratio between 0.1 and 0.8
]

Now, we specify that we would like to optimize for the heating use of the entire facility. We then bundle all of this information together.

In [ ]:
objectives = ["DistrictHeating:Facility"]
problem = Problem(parameters, objectives)

## Sampling

We need to generate samples on which to train a surrogate model.
The sampler uses Latin Hypercube Sampling to distribute the samples in the design space defined by the `Parameters`.
We can check how well distributed they are using `describe()`.

In [ ]:
inputs = sampling.lhs(problem, 10, seed=0)
inputs

## Evaluator
We now define and run the Evaluator, which will run the EnergyPlus instances with the different inputs generated in the previous cell.

In [ ]:
evaluator = EnergyPlusEvaluator(problem, idf, epw_file,
                                out_dir='/tmp/bdopt_runs',   # work files go to the container's own disk (the shared folder is much slower)
                                err_dir='outputdir',         # runs that fail are kept here, with the EnergyPlus error file
                                processes=2)                 # EnergyPlus runs in parallel
t1 = time.time()
train = evaluator.evaluate(inputs, keep_input=True)       # parameters + objective, one row per simulation

t2 = time.time()
train.head()  # first 5 lines

In [ ]:
def niceformat(seconds):
    seconds = seconds % (24 * 3600)
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60
    return hour, minutes, seconds

In [ ]:
time_of_sim = t2 - t1
hours, mins, secs = niceformat(time_of_sim)

print(
    "The total running time: {:2.0f} hours {:2.0f} min {:2.0f} seconds".format(
        hours, mins, secs
    )
)

We can save results at this point:

In [ ]:
train.to_pickle("samples.p")

And load them again later:

In [ ]:
train = pd.read_pickle("samples.p")

## Exploring the training data
Let's look at the relation between the inputs and the output with a bdopt plot: one panel per objective and per parameter.

In [ ]:
viz.pairplot(train, problem, size=3)
plt.show()

Here's a plot of the _design space_ defined by the samples.

In [ ]:
x, y, obj = problem.names()
train.plot.hexbin(x=x, y=y, C=obj, cmap="plasma", sharex=False, gridsize=50)

In [ ]:
problem.names()

## Fitting a surrogate model

Now we can train the model using a scikit-learn pipeline:

In [ ]:
train

In [ ]:
model = pipeline.make_pipeline(StandardScaler(), linear_model.LinearRegression())
model.fit(train[[x, y]].values, train[obj].values)

## Running the surrogate model

We can query the model at a specific point in the design space (insulation thickness 0.2 m, window-to-wall ratio 0.25):

In [ ]:
model.predict([[0.2, 0.25]])

Or for a `dataframe` made from a `dictionary` of points:

In [ ]:
points = pd.DataFrame(data={x: [0.05, 0.25], y: [0.2, 0.6]})
model.predict(points.values)

Next we define a function for querying the surrogate model across the domain at a given density, and plot the result.

In [ ]:
def run_model(model, density):
    p1 = problem.parameters[0].value_descriptor
    a = np.linspace(p1.min, p1.max, density)
    p2 = problem.parameters[1].value_descriptor
    b = np.linspace(p2.min, p2.max, density)
    plot_data = pd.DataFrame(
        np.transpose([np.tile(a, len(b)), np.repeat(b, len(a))]),
        columns=problem.parameter_names,
    )
    return pd.concat([plot_data, pd.Series(model.predict(plot_data.values))], axis=1)


density = 200
df = run_model(model, density)
df.plot.hexbin(x=x, y=y, C=0, cmap="plasma", gridsize=density, sharex=False)

## Interactive surrogate
We make an interactive plot using `ipywidgets`, which executes the model every time a slider is moved.

In [ ]:
# get the min and max of the two variables
(min1, max1), (min2, max2) = (
    (float(p.value_descriptor.min), float(p.value_descriptor.max)) for p in problem.parameters
)
# get the data to plot in the background
df = run_model(model, 100)

# define a wrapper to be queried by interact
def model_wrapper(v1, v2):
    df.plot.hexbin(
        x=x, y=y, C=0, cmap="plasma", gridsize=30, sharex=False
    )  # background plot
    x_lims = plt.xlim()
    y_lims = plt.ylim()
    plt.scatter(x=v1, y=v2)  # plot the marker
    plt.xlim((x_lims))
    plt.ylim((y_lims))
    plt.show()

    value = (
        model.predict([[v1, v2]])[0] / 3.6e6
    )  # find the current value and convert to kWh
    output = (
        "Heating use = " + str(value.round()) + " kWh"
    )  # return the string to display
    return output


# make the interactive plot with two sliders
continuous_update = False
interact(
    model_wrapper,
    v1=FloatSlider(
        min=min1,
        max=max1,
        value=(min1 + max1) / 2,
        description="Thickness [m]",
        step=0.01,
        continuous_update=continuous_update,
    ),
    v2=FloatSlider(
        min=min2,
        max=max2,
        value=(min2 + max2) / 2,
        description="WWR",
        step=0.01,
        continuous_update=continuous_update,
    ),
)

# GP vs Neural Network vs a deeper network

Let's compare now a Gaussian Process with a Neural Network and a deeper network for surrogate model creation. On a more complex design space.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings('ignore', category=ConvergenceWarning)   # the networks do not always "converge" on a handful of samples

In [ ]:
insulation = FieldSelector(class_name='Material', object_name='XPS Extruded Polystyrene  - CO2 Blowing_0041', field_name='Thickness')
insulation_range = RangeParameter(min_val=0.01, max_val=0.45)
insulation_param = Parameter(selector=insulation,
                             value_descriptor=insulation_range,
                             name='Insulation Thickness')

In [ ]:
lights_selector = FieldSelector(class_name='Lights', object_name='*', field_name='Watts per Zone Floor Area')
single_digit_integers = IntegerParameter(8, 12)         # whole numbers of W/m2, 8 to 12 included
lights_param = Parameter(
    lights_selector,
    value_descriptor=single_digit_integers,
    name="Lights Watts/Area",
)

In [ ]:
# one parameter per field of the glazing material named "3"
more_parameters = [
    Parameter(FieldSelector('WindowMaterial:Glazing', '3', 'Conductivity'),
              RangeParameter(0.1, 1), name='Glazing conductivity'),
    Parameter(FieldSelector('WindowMaterial:Glazing', '3', 'Solar Transmittance at Normal Incidence'),
              RangeParameter(0.01, 0.8), name='Glazing solar transmittance'),
]

In [ ]:
window_to_wall = wwr(0.1, 0.8)

In [ ]:
parameters = [insulation_param] + more_parameters + [window_to_wall] + [lights_param]

In [ ]:
objectives = ['Electricity:Facility', 'DistrictHeating:Facility', 'DistrictCooling:Facility']   # EnergyPlus meters, summed over the run period [J]

problem = Problem(parameters, objectives)    # problem = parameters + objectives

In [ ]:
samples = sampling.lhs(problem, 20, seed=0)
samples

In [ ]:
evaluator = EnergyPlusEvaluator(problem, idf, epw_file,
                                out_dir='/tmp/bdopt_runs',
                                err_dir='outputdir',
                                processes=2)

saved = Path('surrogate_simple_simulation.csv')
if saved.exists() and list(pd.read_csv(saved, index_col=0).columns) == problem.names():
    sim_samples = pd.read_csv(saved, index_col=0)          # the simulations were already done and stored
else:
    t1 = time.time()
    sim_samples = evaluator.evaluate(samples, keep_input=True)
    t2 = time.time()
    time_of_sim = t2 - t1
    hours, mins, secs = niceformat(time_of_sim)

    print(
        "The total running time: {:2.0f} hours {:2.0f} min {:2.0f} seconds".format(
            hours, mins, secs
        )
    )
    sim_samples.to_csv(saved)

In [ ]:
sim_samples.columns

In [ ]:
features = list(samples.columns)
print(features)
print(objectives)

In [ ]:
# remove inputs
outputs = sim_samples.drop(features, axis=1)
# outputs

In [ ]:
# remove outputs
inputs = sim_samples.drop(objectives, axis=1)
# inputs

## Train-test split

Next we split the data into a training set (80%) and a testing set (20%).

In [ ]:
train_in, test_in, train_out, test_out = train_test_split(
    inputs, outputs, test_size=0.2, random_state=0
)

## Normalization of inputs

To ensure an equal weighting of inputs and outputs in the backpropagation algorithm fitting the neural network, we have to normalize the input values.
For example the window-to-wall ratio is in the range of 0 to 1 while the lighting power is in a range of 8 to 12 W/m$^2$.
Different options for normalization exist.
Here we bring all features (input variables) to have zero mean and a standard deviation of 1.
Note that we fit the normalizer on training data only, and only *apply* it to the test data.

In [ ]:
scaler = StandardScaler()
train_in_scale = scaler.fit_transform(X=train_in)
test_in_scale = scaler.transform(X=test_in)

scaler_out = StandardScaler()
train_out_scale = scaler_out.fit_transform(X=train_out)

# Gaussian Process


## Hyper-parameters

Before fitting the GP model we define the set of hyperparameters we want to optimize.
Here we use 3 folds in the k-fold cross validation scheme.
We select a set of Kernel functions, which must fit the characteristics of a problem - details and examples may be found in the [Kernel cookbook](https://www.cs.toronto.edu/~duvenaud/cookbook/).
Note that the parameters of the Kernel itself are [optimized during each model fitting run](https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.GaussianProcessRegressor.html).

In [ ]:
hyperparameters = {
    "kernel": [
        None,
        1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-1, 10.0)),
        1.0 * RationalQuadratic(length_scale=1.0, alpha=0.5),
        1.0 * Matern(length_scale=1.0, length_scale_bounds=(1e-1, 10.0)),
    ]
}

folds = 3

## Model fitting

Here we fit the model using these hyperparameters.

In [ ]:
gp = GaussianProcessRegressor(normalize_y=True)

clf = GridSearchCV(gp, hyperparameters, cv=folds)

clf.fit(train_in_scale, train_out_scale)

best_gp = clf.best_estimator_

In [ ]:
prediction_gp = scaler_out.inverse_transform(best_gp.predict(test_in_scale))

# Neural Network

## Hyper-parameters

Before we start fitting the NN model we define the set of hyperparameters we want to analyse in our cross-validation to optimize the model.
Here, we select the number of layers of the network as well as the regularization parameter alpha as parameter value.
A larger number of layers and a lower value of the regularizer lead to higher variance of the network.
This may lead to overfitting.
The best selection may be found using an optimizer like Bayesian Optimization.
In this example we use a simple grid search.

In [ ]:
hyperparameters = {
    "hidden_layer_sizes": (
        (len(parameters) * 16,),
        (len(parameters) * 16, len(parameters) * 16),
    ),
    "alpha": [1, 10, 10 ** 3],
}

neural_net = MLPRegressor(max_iter=1000, early_stopping=False, random_state=0)
folds = 3

## Model fitting

Here, we use the NN model from scikit-learn.

In [ ]:
clf = GridSearchCV(neural_net, hyperparameters, cv=folds)
clf.fit(train_in_scale, train_out_scale)

nn_model = clf.best_estimator_

In [ ]:
prediction_nn = scaler_out.inverse_transform(nn_model.predict(test_in_scale))

## A deeper network

Three hidden layers (32, 64 and 32 neurons, ReLU), trained with the Adam optimizer for 1000 epochs. No deep-learning framework is installed in the course environment: scikit-learn's `MLPRegressor` trains the same architecture. (bdopt also has a `TorchMLP` surrogate, which needs PyTorch.)

In [ ]:
deep_model = MLPRegressor(hidden_layer_sizes=(32, 64, 32), activation='relu', solver='adam',
                          learning_rate_init=0.001, batch_size=min(32, len(train_in_scale)), max_iter=1000, random_state=0)
deep_model.fit(train_in_scale, train_out_scale)

In [ ]:
prediction_deep = scaler_out.inverse_transform(deep_model.predict(test_in_scale))

## Compare the models

RMSE on the test set, as a percentage of the mean of each objective (smaller is better).

In [ ]:
def rmse_percent(prediction):
    return root_mean_squared_error(test_out.values, prediction, multioutput='raw_values') / test_out.mean().values * 100


pd.DataFrame({'GP': rmse_percent(prediction_gp),
              'NN': rmse_percent(prediction_nn),
              'deep NN': rmse_percent(prediction_deep)}, index=objectives)

## Surrogate Modelling Evaluator object
We can wrap the fitted model in a bdopt `FunctionEvaluator`: an evaluator that computes the objectives with a Python function instead of running EnergyPlus. It receives a table of designs, one column per parameter.

In [ ]:
# definition of the evaluation function
def evaluation_func(designs):
    return scaler_out.inverse_transform(nn_model.predict(scaler.transform(designs[features])))


surrogate_evaluator = FunctionEvaluator(problem, evaluation_func, vectorized=True)

In [ ]:
srinputs = sampling.lhs(problem, 500, seed=1)
sroutputs = surrogate_evaluator.evaluate(srinputs)
srresults = srinputs.join(sroutputs)
srresults.head()

## Exploration

Parallel coordinates: one line per design, one axis per parameter and objective (each scaled to 0-1), coloured by the first objective.

In [ ]:
viz.parallel_coordinates(srresults, problem, color='Electricity:Facility')
plt.show()

## Save the surrogate

`Optimization.ipynb` uses a surrogate model stored in a file. bdopt surrogates scale their inputs and outputs inside, so the file is all that is needed (no separate scalers). The one below is the scikit-learn neural network with its grid search, trained on all the simulations.

In [ ]:
from bdopt.surrogate import MLPSurrogate

best_surrogate = MLPSurrogate(random_state=0).fit(sim_samples, problem)
best_surrogate.save('best_surrogate.pkl')